# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaaDasim05/Flyrank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal checks

**Signal 1: `days_since_last_update` — CONFIRMED/MIXED/OPPOSITE/FALSE after the bucket check.**

I am checking whether increasing time since the last update is associated with the refresh-oriented signals in the starter data. This is directly relevant to the staleness logic behind refresh flags.

**Signal 2: `trend_pct` — CONFIRMED/MIXED/OPPOSITE/FALSE after the bucket check.**

I am checking whether stronger negative trend values are associated with content that looks more appropriate for review.

### Baseline rule

I will rank content items using a simple hand-written refresh opportunity score based on:

- content staleness
- negative performance trend

The score is a baseline, not a claim that either signal causes performance changes.

The rule will produce one action label and one reason code for each item.

### Action

High-scoring pages will receive the action label **"Review for refresh"**.

### Reason code

The rule will use exactly one reason code per item:

**STALE_DECLINE** — both staleness and negative trend contribute to the priority.

The baseline exists to create a transparent benchmark that a later ML model must beat.

In [1]:
import pandas as pd
import numpy as np

DATA_URL = "https://raw.githubusercontent.com/SaaDasim05/Flyrank-ML-Internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

print(f"Rows: {len(df):,}")

Rows: 30,000


In [2]:
staleness_bins = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 30, 90, 180, 365, np.inf],
    labels=["0-30d", "31-90d", "91-180d", "181-365d", "365d+"]
)

staleness_check = (
    df.assign(staleness_bucket=staleness_bins)
      .groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          avg_trend_pct=("trend_pct", "mean"),
          down_pct=("trend_direction", lambda s: (s == "down").mean())
      )
      .reset_index()
)

staleness_check

,staleness_bucket,n,avg_trend_pct,down_pct
0,0-30d,20480,0.784405,0.511377
1,31-90d,175,-7.373054,0.588571
2,91-180d,9171,-15.683224,0.611057
3,181-365d,169,-4.718462,0.467456
4,365d+,5,-96.166667,0.600000


In [3]:
trend_bins = pd.cut(
    df["trend_pct"],
    bins=[-np.inf, -50, -20, -5, 5, 20, 50, np.inf],
    labels=[
        "<=-50%",
        "-49% to -20%",
        "-19% to -5%",
        "-4% to +5%",
        "+6% to +20%",
        "+21% to +50%",
        ">+50%"
    ]
)

trend_check = (
    df.assign(trend_bucket=trend_bins)
      .groupby("trend_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          avg_age=("content_age_days", "mean"),
          down_pct=("trend_direction", lambda s: (s == "down").mean())
      )
      .reset_index()
)

trend_check

,trend_bucket,n,avg_age,down_pct
0,<=-50%,9646,217.822517,1.00000
1,-49% to -20%,6667,262.930103,0.99235
2,-19% to -5%,2751,293.981825,0.00000
3,-4% to +5%,1670,294.191617,0.00000
4,+6% to +20%,1492,300.799598,0.00000
5,+21% to +50%,1771,299.570299,0.00000
6,>+50%,2615,280.897897,0.00000


### Signal verdicts

**`days_since_last_update` — MIXED**

The staleness buckets do not show a consistently increasing deterioration pattern. The 91–180 day group has the highest observed down share among the substantial buckets, but the 181–365 day group is lower, and the 365+ group has only five observations. I will therefore use staleness as a secondary prioritization signal rather than treating it as sufficient evidence by itself.

**`trend_pct` — CONFIRMED**

The observed trend buckets show a very strong association with `trend_direction`: strongly negative trend values are overwhelmingly in the `down` category, while non-negative buckets are not. I will use negative trend strength as the stronger component of the baseline score. This should be interpreted as an observed association, not as evidence that the trend signal causes future performance changes.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# ---------------------------------------------------------
# Revised transparent baseline rule
# ---------------------------------------------------------

rule_df = df.copy()

# Negative trend: stronger decline = higher score.
# Cap only at -100%, so normal declines retain differences.
trend_component = (
    (-rule_df["trend_pct"]).clip(lower=0, upper=100) / 100
)

# Staleness: scale over one year instead of capping everything
# above 180 days at the same value.
stale_component = (
    rule_df["days_since_last_update"].clip(lower=0, upper=365) / 365
)

# 60% trend, 40% staleness
rule_df["action_score"] = 100 * (
    0.60 * trend_component +
    0.40 * stale_component
)

# Exactly one reason code
rule_df["reason_code"] = "TREND_OR_STALE"

# Action label
rule_df["action"] = np.where(
    rule_df["action_score"] >= 40,
    "Review for refresh",
    "Monitor"
)

# Rank
queue = (
    rule_df
    .sort_values(
        ["action_score", "trend_pct", "days_since_last_update"],
        ascending=[False, True, False]
    )
    [[
        "content_id",
        "client_id",
        "action_score",
        "reason_code",
        "action",
        "trend_pct",
        "days_since_last_update"
    ]]
    .reset_index(drop=True)
)

queue["rank"] = np.arange(1, len(queue) + 1)

queue.head(20)

,content_id,client_id,action_score,reason_code,action,trend_pct,days_since_last_update,rank
0,content_f6fdf87348f6,client_4ec9599fc2,100.000000,TREND_OR_STALE,Review for refresh,-100.0,373,1
1,content_1b4ec72dafd4,client_4ec9599fc2,100.000000,TREND_OR_STALE,Review for refresh,-100.0,372,2
2,content_7a888d3d99c8,client_19581e27de,94.301370,TREND_OR_STALE,Review for refresh,-100.0,313,3
3,content_94991fe6268c,client_19581e27de,94.301370,TREND_OR_STALE,Review for refresh,-100.0,313,4
4,content_ab18b5811c02,client_19581e27de,93.424658,TREND_OR_STALE,Review for refresh,-100.0,305,5
5,content_84d12054c0c0,client_9400f1b21c,93.315068,TREND_OR_STALE,Review for refresh,-100.0,304,6
6,content_55a5b1c46474,client_4ec9599fc2,93.100000,TREND_OR_STALE,Review for refresh,-88.5,373,7
7,content_ccfb4d0227b1,client_d59eced1de,92.986301,TREND_OR_STALE,Review for refresh,-100.0,301,8
8,content_df1fa766cac2,client_9400f1b21c,91.935068,TREND_OR_STALE,Review for refresh,-97.7,304,9
9,content_02b0d6e30129,client_19581e27de,91.661370,TREND_OR_STALE,Review for refresh,-95.6,313,10


In [8]:
from pathlib import Path

output_path = Path("../outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(output_path, index=False)

print(f"Wrote {len(queue):,} rows to {output_path}")

Wrote 30,000 rows to ../outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
top10 = queue.head(10).copy()

top10[
    [
        "rank",
        "content_id",
        "action_score",
        "action",
        "reason_code",
        "trend_pct",
        "days_since_last_update"
    ]
]

,rank,content_id,action_score,action,reason_code,trend_pct,days_since_last_update
0,1,content_f6fdf87348f6,100.000000,Review for refresh,TREND_OR_STALE,-100.0,373
1,2,content_1b4ec72dafd4,100.000000,Review for refresh,TREND_OR_STALE,-100.0,372
2,3,content_7a888d3d99c8,94.301370,Review for refresh,TREND_OR_STALE,-100.0,313
3,4,content_94991fe6268c,94.301370,Review for refresh,TREND_OR_STALE,-100.0,313
4,5,content_ab18b5811c02,93.424658,Review for refresh,TREND_OR_STALE,-100.0,305
5,6,content_84d12054c0c0,93.315068,Review for refresh,TREND_OR_STALE,-100.0,304
6,7,content_55a5b1c46474,93.100000,Review for refresh,TREND_OR_STALE,-88.5,373
7,8,content_ccfb4d0227b1,92.986301,Review for refresh,TREND_OR_STALE,-100.0,301
8,9,content_df1fa766cac2,91.935068,Review for refresh,TREND_OR_STALE,-97.7,304
9,10,content_02b0d6e30129,91.661370,Review for refresh,TREND_OR_STALE,-95.6,313


In [10]:
from pathlib import Path

output_path = Path("../outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(output_path, index=False)

print(f"Wrote {len(queue):,} rows to {output_path}")

Wrote 30,000 rows to ../outputs/baseline_action_score.csv


### Top-10 review

1. **content_f6fdf87348f6** — **Review for refresh.** It ranks first because the observed trend is -100% and the content has not been updated for 373 days. It could be wrong if the decline reflects a temporary measurement or traffic change rather than a content-quality problem.

2. **content_1b4ec72dafd4** — **Review for refresh.** It combines a -100% trend with 372 days since the last update, producing the maximum score. It could be wrong if the negative trend is caused by an external demand change or another factor unrelated to content freshness.

3. **content_7a888d3d99c8** — **Review for refresh.** Its trend is -100% and it has been 313 days since its last update. It could be wrong if the observed decline is temporary or if the page is intentionally being retired.

4. **content_94991fe6268c** — **Review for refresh.** It has the same strong -100% trend and 313 days since update, placing it near the top of the queue. It could be wrong if the underlying search demand changed independently of the content.

5. **content_ab18b5811c02** — **Review for refresh.** The page has a -100% trend and 305 days since its last update. It could be wrong if the trend measurement reflects a short-lived fluctuation or incomplete data.

6. **content_84d12054c0c0** — **Review for refresh.** Its -100% trend and 304 days since update produce a high priority score. It could be wrong if the page remains strategically important despite the observed decline.

7. **content_55a5b1c46474** — **Review for refresh.** It combines an -88.5% trend with 373 days since the last update. It could be wrong if the content has already become irrelevant to current search demand and a refresh would not address the underlying issue.

8. **content_ccfb4d0227b1** — **Review for refresh.** Its trend is -100% and it has gone 301 days without an update. It could be wrong if the decline came from a change in search behavior rather than from stale content.

9. **content_df1fa766cac2** — **Review for refresh.** It shows a -97.7% trend and 304 days since update, giving it a high score. It could be wrong if the extreme decline is driven by a measurement anomaly or another external factor.

10. **content_02b0d6e30129** — **Review for refresh.** It has a -95.6% trend and 313 days since update. It could be wrong if the observed decline does not represent a recoverable content opportunity.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [11]:
weak_picks = queue.tail(10).copy()

weak_picks[
    [
        "rank",
        "content_id",
        "action_score",
        "action",
        "reason_code",
        "trend_pct",
        "days_since_last_update"
    ]
]

,rank,content_id,action_score,action,reason_code,trend_pct,days_since_last_update
29990,29991,content_b5fb35404aed,NaN,Monitor,TREND_OR_STALE,NaN,1
29991,29992,content_009fe48fadc0,NaN,Monitor,TREND_OR_STALE,NaN,1
29992,29993,content_190cbb83cea5,NaN,Monitor,TREND_OR_STALE,NaN,1
29993,29994,content_00dda64bb29a,NaN,Monitor,TREND_OR_STALE,NaN,1
29994,29995,content_8bce3371c63c,NaN,Monitor,TREND_OR_STALE,NaN,1
29995,29996,content_2a843f006d86,NaN,Monitor,TREND_OR_STALE,NaN,1
29996,29997,content_3a8f5c52b1a0,NaN,Monitor,TREND_OR_STALE,NaN,1
29997,29998,content_94283b065b7c,NaN,Monitor,TREND_OR_STALE,NaN,1
29998,29999,content_9ffe1e2e3575,NaN,Monitor,TREND_OR_STALE,NaN,1
29999,30000,content_0a22a2eeefdd,NaN,Monitor,TREND_OR_STALE,NaN,1


### Weak-pick review

The lowest-scoring items are not necessarily healthy pages. A low score can simply mean that the rule does not observe a combination of strong negative trend and staleness.

This is a limitation of the baseline: it can miss pages whose opportunity is driven by signals that the rule does not consider, such as CTR changes, ranking position, content type, or query-level behavior.

The baseline should therefore be treated as a transparent prioritization benchmark rather than a complete refresh decision system.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.